# 09 - Check of the ESM25GoodMP dataset

ESM25 on SeisBench has four datasets: `ESM25GoodCV` and `ESM25BadCV` (acceleration converted to physical units, the
data of every run so far), `ESM25GoodMP` (manually processed acceleration, velocity and displacement, good records
only) and `ESM25SpectraMP` (response spectra). A classifier trained on processed good records against unprocessed bad
records could separate the classes by the processing (filter band, taper, trimming) rather than by the quality: a
possible explanation of the AUC of 0.9999 of Shalabi et al. (2026).

Before downloading ESM25GoodMP, this notebook checks, without reading any waveform:

1. the size of the waveform files on the server, against the free disk space;
2. the metadata of ESM25GoodMP (downloaded to `data/goodmp/`);
3. how its records match those of ESM25GoodCV (identifiers in both, duplicates) and which columns are specific to the
   processed data (filter corners, processed time window, padding, sampling).

## 0. Setup

In [1]:
from dataclasses import replace
from pathlib import Path
import shutil

import pandas as pd

from wqu.config import load_config
from wqu.data.loading import load_esm25_metadata
from wqu.data.remote_datasets import (
    columns_only_in,
    download_file,
    key_overlap,
    matching_columns,
    remote_dataset_url,
    remote_file_size,
    shared_column_agreement,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
MP_REPOSITORY = "esm25goodmp"
WAVEFORM_FILE = "waveforms.hdf5"
BYTES_PER_GB = 1e9
# Candidate record identifiers shared by the datasets (trace_name may refer to the position inside each file).
KEY_CANDIDATES = ["trace_name_old", "trace_name"]
# Column names that may describe the processing.
PROCESSING_PATTERNS = ["_mp", "filter", "lowcut", "highcut", "corner", "pass", "taper", "padding", "start_time",
                       "end_time", "sampling", "npts", "processing", "baseline"]
# Columns of the trace compared between the two datasets on the matched records.
COMPARED_PATTERNS = ["^trace_start_time$", "^trace_sampling_rate_hz$", "^trace_dt_s$", "^trace_has_padding",
                     "^trace_npts", "^station_code$", "^source_esm_id$"]

config = load_config(CONFIG_PATH)
config = replace(config, data=replace(config.data, source=FULL_SOURCE))
columns = config.columns
template = config.waveforms.remote_url_template
output_dir = config.preprocessing.output_dir.parent / "goodmp"
repositories = {"good MP": MP_REPOSITORY, **{f"{subset.name} CV": subset.repository_name for subset in config.data.subsets}}
print(f"Repositories: {repositories}")
print(f"Output: {output_dir}")

Repositories: {'good MP': 'esm25goodmp', 'good CV': 'esm25goodcv', 'bad CV': 'esm25badcv'}
Output: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/goodmp


/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Size of the files on the server

Free space measured on the disk of the home folder (where the SeisBench cache usually is) and of the project.

In [2]:
sizes = pd.DataFrame({
    name: {file_name: remote_file_size(remote_dataset_url(template, repository, file_name)) / BYTES_PER_GB
           for file_name in (WAVEFORM_FILE, config.data.seisbench_metadata_file)}
    for name, repository in repositories.items()
}).T.rename(columns=lambda name: f"{name} (GB)")
display(sizes.round(2))
for place in (Path.home(), output_dir.parent.resolve()):
    print(f"Free space on the disk of {place}: {shutil.disk_usage(place).free / BYTES_PER_GB:.1f} GB")

,waveforms.hdf5 (GB),metadata.csv (GB)
good MP,194.18,0.39
good CV,64.73,0.39
bad CV,36.32,0.03


Free space on the disk of /Users/giulianaparadiso: 34.0 GB
Free space on the disk of /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data: 34.0 GB


## 2. Metadata of ESM25GoodMP

In [3]:
mp_metadata_path = output_dir / config.data.seisbench_metadata_file
if not mp_metadata_path.exists():
    download_file(remote_dataset_url(template, MP_REPOSITORY, config.data.seisbench_metadata_file), mp_metadata_path)
text_columns = {column: str for column in columns.text}
mp_metadata = pd.read_csv(mp_metadata_path, dtype=text_columns, low_memory=False)
print(f"ESM25GoodMP: {len(mp_metadata):,} records, {mp_metadata.shape[1]} columns")
mp_metadata.head(3).T.head(40)

metadata.csv: 100%|██████████| 385M/385M [00:46<00:00, 8.24MB/s] 


ESM25GoodMP: 64,213 records, 598 columns


,0,1,2
source_esm_id,AL-2014-0005,AL-2014-0005,AL-2014-0005
source_esm_origin_time,2014-01-20T14:40:48,2014-01-20T14:40:48,2014-01-20T14:40:48
source_isc_id,609700745.0,609700745.0,609700745.0
source_usgs_id,usb000m4t2,usb000m4t2,usb000m4t2
source_ingv_id,2986911.0,2986911.0,2986911.0
source_emsc_id,20140120_0000089,20140120_0000089,20140120_0000089
source_nation_code,AL,AL,AL
source_latitude_deg,41.38,41.38,41.38
source_longitude_deg,19.45,19.45,19.45
source_depth_km,8.0,8.0,8.0


## 3. Records and columns compared with ESM25GoodCV

Identifiers in both datasets for each candidate key: the key with all the records in both and no duplicates is the one
to use to pair the processed and unprocessed version of each record.

In [4]:
_, metadata = load_esm25_metadata(config)
good_cv = metadata[metadata[columns.label] == 0].reset_index(drop=True)
overlap = pd.DataFrame({
    key: key_overlap(mp_metadata[key], good_cv[key], ("good MP", "good CV"))
    for key in KEY_CANDIDATES if key in mp_metadata.columns and key in good_cv.columns
})
overlap

,trace_name_old,trace_name
records in good MP,64213,64213
records in good CV,64213,64213
identifiers in both,64213,64213
only in good MP,0,0
only in good CV,0,0
duplicated in good MP,0,0
duplicated in good CV,0,0


In [5]:
only_mp = columns_only_in(mp_metadata, good_cv)
only_cv = columns_only_in(good_cv, mp_metadata)
print(f"Columns only in good MP ({len(only_mp)}): {only_mp}")
print(f"Columns only in good CV ({len(only_cv)}): {only_cv}")
processing_columns = matching_columns(mp_metadata.columns, PROCESSING_PATTERNS)
print(f"Columns of good MP that may describe the processing ({len(processing_columns)}):")
mp_metadata[processing_columns].describe(include="all").T

Columns only in good MP (0): []
Columns only in good CV (13): ['preferred_magnitude', 'preferred_magnitude_type', 'original_duration_s', 'recording_span_s', 'padding_fraction', 'component_start_spread_s', 'original_sampling_rate_hz', 'origin_year', 'max_unprocessed_pga_cps2', 'station_id', 'subset_index', 'subset', 'label']
Columns of good MP that may describe the processing (23):


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
station_preferred_vs30_mps,47117.0,NaN,NaN,NaN,594.640414,287.155324,90.0,397.0,514.0,753.0,3019.0
station_vs30_mps,12248.0,NaN,NaN,NaN,565.783067,335.163032,90.0,321.0,486.0,705.0,2925.0
station_vs30_mps_wa,61626.0,NaN,NaN,NaN,632.248893,281.037604,100.0,407.0,573.0,823.2,3019.0
trace_processing_status,64213,1,processed,64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trace_processing_type,64213,1,manual (Paolucci et al 2011),64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trace_npts_cv_u,64213.0,NaN,NaN,NaN,36246.267173,18154.552977,469.0,26250.0,30634.0,53140.0,176651.0
trace_npts_cv_v,64213.0,NaN,NaN,NaN,36245.132092,18154.328903,469.0,26256.0,30623.0,53142.0,176416.0
trace_npts_cv_w,64213.0,NaN,NaN,NaN,36239.197701,18150.033081,469.0,26248.0,30639.0,53134.0,176694.0
trace_has_padding_cv,64213,1,True,64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trace_sampling_rate_hz,64213.0,NaN,NaN,NaN,200.0,0.0,200.0,200.0,200.0,200.0,200.0


Agreement of the trace columns present in both datasets, on the records matched by the chosen key: equal start
time, sampling rate and padding flags mean that the processed traces are stored on the same time axis as the
unprocessed ones.

In [6]:
key = overlap.loc["identifiers in both"].idxmax()
shared = [column for column in matching_columns(mp_metadata.columns, COMPARED_PATTERNS) if column in good_cv.columns]
print(f"Key: {key}; compared columns: {shared}")
shared_column_agreement(mp_metadata, good_cv, key, shared).round(4)

Key: trace_name_old; compared columns: ['source_esm_id', 'station_code', 'trace_npts_cv_u', 'trace_npts_cv_v', 'trace_npts_cv_w', 'trace_has_padding_cv', 'trace_sampling_rate_hz', 'trace_dt_s', 'trace_start_time', 'trace_has_padding_mp', 'trace_npts_mp']


,matched_records,share_equal
source_esm_id,64213,1.0
station_code,64213,1.0
trace_npts_cv_u,64213,1.0
trace_npts_cv_v,64213,1.0
trace_npts_cv_w,64213,1.0
trace_has_padding_cv,64213,1.0
trace_sampling_rate_hz,64213,1.0
trace_dt_s,64213,1.0
trace_start_time,64213,1.0
trace_has_padding_mp,64213,1.0


In [1]:
from pathlib import Path

import pandas as pd

from wqu.config import load_config
from wqu.data.loading import load_esm25_metadata

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
# Parts of the column names that may describe the network, the channel or the country of the station.
COLUMN_PATTERNS = ("network", "channel", "instrument", "country")
NETWORK_COLUMN = "station_network_code"
# Rows shown per table and distinct values above which a column is summarised instead of tabulated.
MAX_ROWS = 30
MAX_TABULATED_VALUES = 200


def candidate_columns(table: pd.DataFrame, patterns: tuple[str, ...]) -> list[str]:
    """Names of the columns that contain any of the patterns (case-insensitive).

    Args:
        table: Metadata table.
        patterns: Parts of column names to look for.

    Returns:
        Matching column names, in table order.
    """
    return [name for name in table.columns if any(pattern in name.lower() for pattern in patterns)]


def class_counts(table: pd.DataFrame, column: str, label_column: str, class_names: dict[int, str]) -> pd.DataFrame:
    """Number of records per value of a column and class, with the total and the fraction of bad records.

    Args:
        table: Metadata table with a label column.
        column: Column whose values are counted (missing values counted as "missing").
        label_column: Column holding the class label.
        class_names: Mapping from label to class name.

    Returns:
        Table indexed by value, sorted by decreasing total.
    """
    values = table[column].astype("string").fillna("missing")
    counts = pd.crosstab(values, table[label_column].map(class_names))
    counts["total"] = counts.sum(axis=1)
    counts["bad fraction"] = (counts.get("bad", 0) / counts["total"]).round(3)
    return counts.sort_values("total", ascending=False)


config = load_config(CONFIG_PATH)
columns = config.columns
class_names = {subset.label: subset.name for subset in config.data.subsets}
_, metadata = load_esm25_metadata(config)
print(f"Records: {len(metadata):,}; classes: {class_names}")

found = candidate_columns(metadata, COLUMN_PATTERNS)
summary = pd.DataFrame({
    "distinct values": metadata[found].nunique(),
    "missing": metadata[found].isna().sum(),
    "examples": [", ".join(map(str, metadata[name].dropna().unique()[:6])) for name in found],
})
display(summary)

for name in found:
    if metadata[name].nunique() <= MAX_TABULATED_VALUES:
        print(f"\n{name}")
        display(class_counts(metadata, name, columns.label, class_names).head(MAX_ROWS))

if NETWORK_COLUMN in metadata.columns:
    networks = class_counts(metadata, NETWORK_COLUMN, columns.label, class_names)
    print(f"\n{len(networks)} networks; the {MAX_ROWS} largest cover "
          f"{networks['total'].head(MAX_ROWS).sum() / len(metadata):.1%} of the records")

Records: 100,249; classes: {0: 'good', 1: 'bad'}


,distinct values,missing,examples
station_network_code,85,0,"AC, HL, HI, EG, CL, A"
station_channel_code,7,0,"HN, EN, HG, HL, DH, HH"
station_instrument_type_code,3,0,"D, A, U"
trace_u_channel_code,3,0,"E, 1, 2"
trace_v_channel_code,3,0,"N, 2, 3"
trace_w_channel_code,2,0,"Z, 3"



station_network_code


label,bad,good,total,bad fraction
station_network_code,,,,
HL,8067,12160,20227,0.399
IT,3639,12168,15807,0.230
IV,5506,7678,13184,0.418
HI,4736,7523,12259,0.386
RO,3262,4388,7650,0.426
KO,3909,3541,7450,0.525
HP,972,2267,3239,0.300
CH,713,1888,2601,0.274
RA,558,1404,1962,0.284



station_channel_code


label,bad,good,total,bad fraction
station_channel_code,,,,
HN,32453,53597,86050,0.377
HG,2993,9471,12464,0.240
HL,125,769,894,0.140
EN,375,255,630,0.595
DH,73,114,187,0.390
HH,15,7,22,0.682
EH,2,0,2,1.000


... (5 outputs left out)

label,bad,good,total,bad fraction
trace_v_channel_code,,,,
N,35229,62820,98049,0.359
2,549,596,1145,0.479
3,258,797,1055,0.245



trace_w_channel_code


label,bad,good,total,bad fraction
trace_w_channel_code,,,,
Z,35661,63959,99620,0.358
3,375,254,629,0.596



85 networks; the 30 largest cover 97.8% of the records


In [2]:
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd

from wqu.config import CV_GOOD_SOURCE, load_config
from wqu.preprocessing.features import representation_feature_path
from wqu.preprocessing.pipeline import read_feature_records
from wqu.training.metrics import classification_metrics
from wqu.training.trainer import VALIDATION_PREDICTIONS_FILE, find_run_dir

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
RUN_NAME = "obspy_spectrogram_20261124_190536"
PREDICTION_COLUMN = "probability_deterministic"
NETWORK_COLUMN = "station_network_code"
CHANNEL_COLUMN = "station_channel_code"
ORIENTATION_COLUMNS = {"trace_u_channel_code": "E", "trace_v_channel_code": "N", "trace_w_channel_code": "Z"}
SHALABI_CHANNEL = "HN"
# Networks of the archives named by Shalabi et al.: ITACA (Italy), HEAD (Greece), TR-NSMN (Turkey).
BROAD_NETWORKS = ("IT", "IV", "HL", "HI", "HP", "HA", "TK")
STRICT_NETWORKS = ("IT", "HL", "HI", "TK")
# Groups with fewer validation records of either class are left out of the per-group tables.
MIN_RECORDS_PER_CLASS = 30


def shalabi_channel_mask(records: pd.DataFrame) -> pd.Series:
    """Records with HN channels and E, N, Z orientations, as the HNE, HNN, HNZ files of Shalabi et al.

    Args:
        records: Record table with the channel and orientation columns.

    Returns:
        Boolean mask aligned with the table.
    """
    mask = records[CHANNEL_COLUMN] == SHALABI_CHANNEL
    for column, orientation in ORIENTATION_COLUMNS.items():
        mask &= records[column] == orientation
    return mask


def subset_metrics(probabilities: np.ndarray, labels: np.ndarray, n_bins: int) -> dict[str, float]:
    """Size, bad fraction, AUC and balanced accuracy of a set of records.

    Args:
        probabilities: Probabilities of the bad class.
        labels: Labels (1 = bad).
        n_bins: Bins of the calibration error (required by ``classification_metrics``).

    Returns:
        Dictionary with ``records``, ``bad_fraction``, ``auc`` and ``balanced_accuracy``.
    """
    values = classification_metrics(probabilities, labels, n_bins)
    return {"records": len(labels), "bad_fraction": float(labels.mean()),
            "auc": values["auc"], "balanced_accuracy": values["balanced_accuracy"]}


def group_metrics(probabilities: np.ndarray, labels: np.ndarray, groups: pd.Series, n_bins: int,
                  min_per_class: int) -> pd.DataFrame:
    """Metrics of every group with at least ``min_per_class`` records of each class.

    Args:
        probabilities: Probabilities of the bad class.
        labels: Labels (1 = bad).
        groups: Group of every record, aligned with the probabilities.
        n_bins: Bins of the calibration error.
        min_per_class: Minimum number of good and of bad records of a group.

    Returns:
        Table indexed by group, sorted by decreasing number of records.
    """
    rows = {}
    for group in groups.unique():
        inside = (groups == group).to_numpy()
        n_bad = int(labels[inside].sum())
        if min(n_bad, int(inside.sum()) - n_bad) >= min_per_class:
            rows[group] = subset_metrics(probabilities[inside], labels[inside], n_bins)
    return pd.DataFrame.from_dict(rows, orient="index").sort_values("records", ascending=False)


config = load_config(CONFIG_PATH)
label_column = config.columns.label
n_bins = config.training.calibration_bins
obspy_settings = replace(config.preprocessing.obspy_spectrogram, good_source=CV_GOOD_SOURCE)
preprocessing = replace(config.preprocessing, obspy_spectrogram=obspy_settings)
features_path = representation_feature_path(preprocessing, config.training.feature_source,
                                            config.training.representation)

run_dir = find_run_dir(config.training.runs_dir, config.training.representation, RUN_NAME)
predictions = pd.read_csv(run_dir / VALIDATION_PREDICTIONS_FILE)
records = read_feature_records(features_path, config.columns.text)
required = [NETWORK_COLUMN, CHANNEL_COLUMN, *ORIENTATION_COLUMNS]
missing = [column for column in required if column not in records.columns]
if missing:
    raise KeyError(f"Columns missing from {features_path.parent / 'records.csv'}: {missing}")
records = records.loc[predictions["record"].to_numpy(), required].reset_index(drop=True)

probabilities = predictions[PREDICTION_COLUMN].to_numpy()
labels = predictions[label_column].to_numpy()
channel_mask = shalabi_channel_mask(records)
subsets = {
    "all": pd.Series(True, index=records.index),
    "HN, E/N/Z": channel_mask,
    "broad networks": records[NETWORK_COLUMN].isin(BROAD_NETWORKS),
    "broad networks, HN, E/N/Z": records[NETWORK_COLUMN].isin(BROAD_NETWORKS) & channel_mask,
    "strict networks, HN, E/N/Z": records[NETWORK_COLUMN].isin(STRICT_NETWORKS) & channel_mask,
}
subset_table = pd.DataFrame({
    name: subset_metrics(probabilities[mask.to_numpy()], labels[mask.to_numpy()], n_bins)
    for name, mask in subsets.items()
}).T
print(f"Run: {run_dir.name}; validation records: {len(predictions):,}")
display(subset_table.round(4))

network_table = group_metrics(probabilities, labels, records[NETWORK_COLUMN], n_bins, MIN_RECORDS_PER_CLASS)
weighted_auc = np.average(network_table["auc"], weights=network_table["records"])
print(f"\nPer network ({len(network_table)} networks with at least {MIN_RECORDS_PER_CLASS} records per class); "
      f"record-weighted mean of the within-network AUC: {weighted_auc:.4f}")
display(network_table.round(4))

print("\nPer channel code")
display(group_metrics(probabilities, labels, records[CHANNEL_COLUMN], n_bins, MIN_RECORDS_PER_CLASS).round(4))

/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run: obspy_spectrogram_20261124_190536; validation records: 20,024


,records,bad_fraction,auc,balanced_accuracy
all,20024.0,0.3586,0.9225,0.8359
"HN, E/N/Z",17001.0,0.3767,0.9188,0.8329
broad networks,13358.0,0.3523,0.9227,0.8352
"broad networks, HN, E/N/Z",11325.0,0.3722,0.9155,0.8288
"strict networks, HN, E/N/Z",8004.0,0.3634,0.9141,0.8288



Per network (18 networks with at least 30 records per class); record-weighted mean of the within-network AUC: 0.9184


,records,bad_fraction,auc,balanced_accuracy
HL,4019,0.4026,0.8872,0.8038
IT,3151,0.2390,0.9476,0.8574
IV,2668,0.4104,0.9148,0.8281
HI,2470,0.3741,0.9376,0.8520
RO,1515,0.4284,0.9212,0.8331
KO,1501,0.5276,0.9310,0.8552
HP,595,0.2891,0.9331,0.8238
CH,517,0.2824,0.8740,0.7858
RA,393,0.2723,0.8921,0.7828
TK,362,0.2403,0.8852,0.8042



Per channel code


,records,bad_fraction,auc,balanced_accuracy
HN,17270,0.3750,0.9185,0.8323
HG,2439,0.2460,0.9401,0.8507
EN,108,0.6019,0.9617,0.8685
